# Amazon ML Challenge 2026: Business Entity Resolution

Run the cells **from top to bottom**, one at a time, using Shift+Enter. Wait for each cell to finish (its ▶ icon stops spinning) before running the next.

If Colab disconnects, run cells 1–4 again, then continue from the stage that didn't finish.

## Cell 1: Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Unzip the dataset (skipped automatically if it's already there)

In [ ]:
import os, shutil
ZIP_PATH = "/content/drive/MyDrive/Amazon ML Challenge/MLdataSet.zip"
if not os.path.isdir("/content/work/student_resource/dataset/train"):
    !unzip -q -o "{ZIP_PATH}" -d /content/work
shutil.rmtree("/content/work/__MACOSX", ignore_errors=True)
%cd /content/work/student_resource
!ls dataset/train dataset/test

## Cell 3: Install the libraries

In [ ]:
!pip install -q rapidfuzz lightgbm indic-transliteration

## Cell 4: Unzip our code into the project folder

In [ ]:
!unzip -q -o "/content/drive/MyDrive/Amazon ML Challenge/business_entity_resolution.zip" -d /content/work/student_resource/code
PIPE = "python code/business_entity_resolution/src/pipeline.py"
DRIVE = "/content/drive/MyDrive/Amazon ML Challenge"
COMMON = f'--data dataset --work /content/cache --model-dir "{DRIVE}/model" --out output'
!ls code/business_entity_resolution/src

## Cell 5: PREP — read and clean all 7 files (about 20 min; skipped if already done in this session)

In [ ]:
if os.path.exists("/content/cache/test_s23.parquet"):
    print("prep already done in this session - skipping")
else:
    !{PIPE} prep {COMMON}

## Cell 6: BLOCKING — find candidate pairs for train and test (about 40–50 min in v4)

In [ ]:
# v4 changed blocking, so blocking must be redone once (delete old candidate files first)
import glob
if not os.path.exists("/content/cache/v4_block_done"):
    for f in glob.glob("/content/cache/*_cand.npz"): os.remove(f)
    !{PIPE} block {COMMON}
    if os.path.exists("/content/cache/test_cand.npz"): open("/content/cache/v4_block_done", "w").close()
else:
    print("v4 blocking already done in this session - skipping")

## Cell 7: TRAIN — stage-1 pruning model + stage-2 matching model (about 35–45 min)

**Copy the output of this cell into the chat.**

In [ ]:
!{PIPE} train {COMMON}

## Cell 8: PREDICT — prune, score and write both TSV files (about 15–20 min). **Copy this output too.**

In [ ]:
!{PIPE} predict {COMMON}

## Cell 9: Validate the output (must say PASS)

In [ ]:
!python utils/validate_submission.py --matching output/matching_results.tsv --candidate output/candidate_pairs.tsv --test-dir dataset/test

## Cell 10: Back up the outputs to Drive and download matching_results.tsv

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%m%d_%H%M")
dest = f"{DRIVE}/submissions/{stamp}"
os.makedirs(dest, exist_ok=True)
!cp output/matching_results.tsv output/candidate_pairs.tsv "{dest}/"
!cp "{DRIVE}/model/model_meta.json" "{DRIVE}/model/run_log.txt" "{dest}/"
print("saved to", dest)
from google.colab import files
files.download("output/matching_results.tsv")

## Cell 11: Save the exact library versions (needed for the final zip)

In [ ]:
!pip freeze | grep -i -E "^(numpy|pandas|pyarrow|rapidfuzz|lightgbm|indic.transliteration)==" | tee "{DRIVE}/requirements_colab.txt"